In [1]:
# If needed:
#   pip install yfinance pandas fastparquet    # easiest path
#   # or: pip install yfinance pandas pyarrow

import warnings
from pathlib import Path
import pandas as pd
import yfinance as yf

TICKER = "SPY"                     # SPDR S&P 500 ETF Trust
START  = "1993-01-01"
OUTDIR = Path("data_spy_yahoo")
OUTDIR.mkdir(exist_ok=True)

# 1) Raw OHLCV (unadjusted)
df = yf.download(
    TICKER,
    start=START,
    interval="1d",
    auto_adjust=False,
    progress=False,
    threads=True
)
df = df[~df.index.duplicated(keep="last")]

# 2) Adjusted series (dividends & splits applied to price columns)
df_adj = yf.download(
    TICKER,
    start=START,
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=True
)
df_adj = df_adj[~df.index.duplicated(keep="last")]

# 3) Corporate actions
t = yf.Ticker(TICKER)
dividends = t.dividends
splits    = t.splits

# 4) Helpful return column (from adjusted close)
df_adj_returns = df_adj.copy()
df_adj_returns["Return"] = df_adj_returns["Close"].pct_change()

# --- Helper: robust Parquet writer with fallbacks ---
def safe_to_parquet(df_in: pd.DataFrame, path: Path):
    # Try fastparquet first (avoids pyarrow extension clashes)
    try:
        import fastparquet  # noqa: F401
        df_in.to_parquet(path, engine="fastparquet", index=True)
        return
    except Exception as e_fast:
        last_err = e_fast

    # Try pyarrow with extension unregister hotfix
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq

        # Hotfix for pandas<->pyarrow extension registration clashes
        for _name in ("arrow.py_extension_type", "pandas.period", "pandas.interval"):
            try:
                pa.unregister_extension_type(_name)
            except Exception:
                pass

        # First attempt via pandas -> pyarrow
        try:
            df_in.to_parquet(path, engine="pyarrow", index=True)
            return
        except Exception:
            # Fallback: call pyarrow directly
            tbl = pa.Table.from_pandas(df_in.reset_index())
            pq.write_table(tbl, path)
            return
    except Exception as e_arrow:
        last_err = e_arrow

    # Final fallback: CSV
    warnings.warn(
        f"Parquet write failed ({type(last_err).__name__}: {last_err}). "
        "Saved CSV instead."
    )
    df_in.to_csv(path.with_suffix(".csv"), index=True)

# 5) Save to disk
df.to_csv(OUTDIR / "SPY_yahoo_ohlcv_raw.csv")
df_adj.to_csv(OUTDIR / "SPY_yahoo_ohlcv_adjusted.csv")
dividends.to_csv(OUTDIR / "SPY_dividends.csv")
splits.to_csv(OUTDIR / "SPY_splits.csv")

safe_to_parquet(df, OUTDIR / "SPY_yahoo_ohlcv_raw.parquet")
safe_to_parquet(df_adj, OUTDIR / "SPY_yahoo_ohlcv_adjusted.parquet")

# 6) Quick sanity prints
print("Raw OHLCV:", df.index.min(), "→", df.index.max(), "| rows:", len(df))
print("Adj OHLCV:", df_adj.index.min(), "→", df_adj.index.max(), "| rows:", len(df_adj))
print("Dividends:", dividends.index.min(), "→", dividends.index.max(), "| rows:", len(dividends))
print("Splits:   ", splits.index.min(), "→", splits.index.max(), "| rows:", len(splits))


Raw OHLCV: 1993-01-29 00:00:00 → 2025-11-03 00:00:00 | rows: 8248
Adj OHLCV: 1993-01-29 00:00:00 → 2025-11-03 00:00:00 | rows: 8248
Dividends: 1993-03-19 00:00:00-05:00 → 2025-09-19 00:00:00-04:00 | rows: 132
Splits:    NaT → NaT | rows: 0
